#Create a parity drop function


In [106]:
import numpy as np
import random

def parity_drop_from_string_RANDOM(password):
    """
    drops 8 RANDOM bits → gives a random 56-bit key order!
    """
    # Step 1: Check the password
    if len(password) != 8:
        print("Oops! Password must be 8 letters! Like 'sigmaboy'")
        return None

    # Step 2: Turn 8 letters into 64 bits
    bits = []
    for char in password:
        binary_char = format(ord(char), '08b')  # 8 bits per letter
        bits.extend([int(b) for b in binary_char])

    # Step 3: Pick 8 RANDOM spots to drop (like picking 8 toys to hide!)
    all_positions = list(range(64))  # 0 to 63
    drop_these = random.sample(all_positions, 8)  # Pick 8 random ones!

    # Step 4: Keep only the bits NOT in the drop list
    key_56 = []
    for i in range(64):
        if i not in drop_these:
            key_56.append(bits[i])

    # Step 5: Turn into a fancy 1x56 box
    return np.array(key_56, dtype=np.uint8).reshape(1, 56), drop_these

# Example: 8-character password
pwd = "sigmaboy"
matrix, dropped = parity_drop_from_string_RANDOM(pwd)

print(f"Password: '{pwd}'")
print(f"Dropped these 8 random spots: {dropped}")
print(f"Your secret 56-bit key:\n{matrix}")


Password: 'sigmaboy'
Dropped these 8 random spots: [4, 53, 23, 37, 35, 9, 27, 8]
Your secret 56-bit key:
[[0 1 1 1 0 1 1 1 0 1 0 0 1 0 1 1 0 0 1 1 0 1 1 1 1 0 1 0 1 1 0 0 1 0 1 1
  0 0 0 1 0 0 1 1 0 1 1 1 0 1 1 1 1 0 0 1]]


#Create shift functions


In [109]:
def shift_once(C, D):
    """
    Performs one right circular shift on both 28-bit halves (C and D).

    Parameters:
        C (np.ndarray): 1x28 row vector (left half)
        D (np.ndarray): 1x28 row vector (right half)

    Returns:
        tuple: (C_shifted, D_shifted) as 1x28 NumPy arrays
    """
    if C.shape != (1, 28) or D.shape != (1, 28):
        raise ValueError("C and D must be 1x28 NumPy arrays")

    # right shift by 1: move first bit to the end
    C_shifted = np.roll(C, 1)  # roll right by 1
    D_shifted = np.roll(D, 1)

    return C_shifted, D_shifted


def shift_twice(C, D):
    """
    Performs two right circular shifts on both 28-bit halves (C and D).

    Parameters:
        C (np.ndarray): 1x28 row vector
        D (np.ndarray): 1x28 row vector

    Returns:
        tuple: (C_shifted, D_shifted) as 1x28 NumPy arrays
    """
    if C.shape != (1, 28) or D.shape != (1, 28):
        raise ValueError("C and D must be 1x28 NumPy arrays")

    # right shift by 2
    C_shifted = np.roll(C, 2)
    D_shifted = np.roll(D, 2)

    return C_shifted, D_shifted



#Create a compression_pbox function

In [110]:
def compression_pbox(combined_1x56):
    # 1. shape check
    if combined_1x56.shape != (1, 56):
        raise ValueError("Input must be a 1×56 matrix")

    # 2. parity-bit positions to drop (0-indexed)
    drop_indices = {7, 15, 23, 31, 39, 47, 51, 21}

    # 3. keep everything *except* the drop indices
    key_48 = [bit for i, bit in enumerate(combined_1x56[0]) if i not in drop_indices]

    # 4. reshape to (1, 48) and force uint8 dtype
    key_flat = np.array(key_48, dtype=np.uint8).reshape(1, 48)

    return key_flat

# --- cell 2 --------------------------------------------------------------
key = compression_pbox(combined)
print("Input shape :", combined.shape)
print("Output shape:", key.shape)
print("48-bit key  :\n", key)

Input shape : (1, 56)
Output shape: (1, 48)
48-bit key  :
 [[0 0 1 1 0 0 1 0 1 1 0 1 0 1 1 1 0 0 1 1 1 0 1 1 1 1 1 0 0 0 1 0 1 1 0 0
  1 0 0 1 1 1 1 0 1 1 0 0]]


#Loop the compression_pbox to get the keys

In [111]:
key_56, dropped_positions = parity_drop_from_string_RANDOM(pwd)          # 1×56
print(f"Password: {pwd}")
print(f"56-bit key (1×56):\n{key_56}\n")

# --------------------------------------------------------------
# Split into C0 / D0
# --------------------------------------------------------------
C = key_56[:, :28].copy()      # bits 1-28
D = key_56[:, 28:].copy()      # bits 29-56
print(f"C0 (1×28):\n{C}")
print(f"D0 (1×28):\n{D}\n")

# --------------------------------------------------------------
# DES shift schedule (how many bits to shift each round)
# --------------------------------------------------------------
shift_table = [1,1,2,2,2,2,2,2,1,2,2,2,2,2,2,2]   # 16 entries

# --------------------------------------------------------------
# Store every round
# --------------------------------------------------------------
all_rounds = []                     # list of dicts, one per round

for rnd in range(1, 17):            # rounds 1 … 16
    # ---- apply the correct number of shifts ----
    if shift_table[rnd-1] == 1:
        C, D = shift_once(C, D)
    else:                           # == 2
        C, D = shift_twice(C, D)

    # ---- combine back to a 1×56 block (just for printing) ----
    combined = np.column_stack((C, D))
    key_matrix = compression_pbox(combined)
    key = ''.join(map(str, key_matrix[0]))

    # ---- save everything ----
    all_rounds.append({
        "round"     : rnd,
        "C"         : C.copy(),
        "D"         : D.copy(),
        "combined"  : combined.copy(),
        "key_matrix"       : key_matrix.copy()
    })

    # ---- pretty-print the current round ----
    print(f"\n=== ROUND {rnd} (shift {shift_table[rnd-1]} bit{'s' if shift_table[rnd-1]>1 else ''}) ===")
    print(f"C{rnd}:\n{C}")
    print(f"D{rnd}:\n{D}")
    print(f"Combined{rnd} (1×56):\n{combined}")
    print(f"key{rnd} (1x48):\n{key_matrix}")
    print(f"key{rnd} string:\n{key}")

# --------------------------------------------------------------
# Quick access to any round (example)
# --------------------------------------------------------------
print("\n--- Example: round 9 ---")
print(all_rounds[8]["key_matrix"])      # index 8 → round 9

Password: sigmaboy
56-bit key (1×56):
[[0 1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 0 1 1 0 0 0 0 1
  0 1 1 0 0 0 1 0 0 1 0 1 1 1 0 1 1 0 0 1]]

C0 (1×28):
[[0 1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1]]
D0 (1×28):
[[0 1 1 0 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 1 1 0 1 1 0 0 1]]


=== ROUND 1 (shift 1 bit) ===
C1:
[[1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1 0]]
D1:
[[1 0 1 1 0 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 1 1 0 1 1 0 0]]
Combined1 (1×56):
[[1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1 0 1 0 1 1 0 0 0 0
  1 0 1 1 0 0 0 1 0 0 1 0 1 1 1 0 1 1 0 0]]
key1 (1x48):
[[1 0 1 1 0 1 1 1 1 0 1 0 0 1 1 1 0 1 1 1 0 1 1 0 1 0 1 0 0 0 0 1 0 1 0 0
  0 1 0 0 1 1 1 1 1 1 0 0]]
key1 string:
101101111010011101110110101000010100010011111100

=== ROUND 2 (shift 1 bit) ===
C2:
[[0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 0 1 1 0 1 1 0 1 1]]
D2:
[[0 1 0 1 1 0 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 1 1 0 1 1 0]]
Combined2 (1×56):
[[0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 1 0 1 1 0 1 1 0 1 1